In [4]:
import pandas as pd
import numpy as np

In [7]:
df = pd.read_csv('C:/Users/aglaf/Documents/data engineering passion projects/projects/healthcare readmission analysis/data/raw/diabetic_data.csv')

In [8]:
df.head()
df.shape
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      101766 non-null  object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   weight                    101766 non-null  object
 6   admission_type_id         101766 non-null  int64 
 7   discharge_disposition_id  101766 non-null  int64 
 8   admission_source_id       101766 non-null  int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                101766 non-null  object
 11  medical_specialty         101766 non-null  object
 12  num_lab_procedures        101766 non-null  int64 
 13  num_procedures            101766 non-null  int64 
 14  num_

encounter_id                    0
patient_nbr                     0
race                            0
gender                          0
age                             0
weight                          0
admission_type_id               0
discharge_disposition_id        0
admission_source_id             0
time_in_hospital                0
payer_code                      0
medical_specialty               0
num_lab_procedures              0
num_procedures                  0
num_medications                 0
number_outpatient               0
number_emergency                0
number_inpatient                0
diag_1                          0
diag_2                          0
diag_3                          0
number_diagnoses                0
max_glu_serum               96420
A1Cresult                   84748
metformin                       0
repaglinide                     0
nateglinide                     0
chlorpropamide                  0
glimepiride                     0
acetohexamide 

In [9]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [10]:
df.describe()

,encounter_id,patient_nbr,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses
count,1.017660e+05,1.017660e+05,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000
mean,1.652016e+08,5.433040e+07,2.024006,3.715642,5.754437,4.395987,43.095641,1.339730,16.021844,0.369357,0.197836,0.635566,7.422607
std,1.026403e+08,3.869636e+07,1.445403,5.280166,4.064081,2.985108,19.674362,1.705807,8.127566,1.267265,0.930472,1.262863,1.933600
min,1.252200e+04,1.350000e+02,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000
25%,8.496119e+07,2.341322e+07,1.000000,1.000000,1.000000,2.000000,31.000000,0.000000,10.000000,0.000000,0.000000,0.000000,6.000000
50%,1.523890e+08,4.550514e+07,1.000000,1.000000,7.000000,4.000000,44.000000,1.000000,15.000000,0.000000,0.000000,0.000000,8.000000
75%,2.302709e+08,8.754595e+07,3.000000,4.000000,7.000000,6.000000,57.000000,2.000000,20.000000,0.000000,0.000000,1.000000,9.000000
max,4.438672e+08,1.895026e+08,8.000000,28.000000,25.000000,14.000000,132.000000,6.000000,81.000000,42.000000,76.000000,21.000000,16.000000


In [11]:
df['readmitted'].value_counts()

readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64

How many rows/columns?
Which columns have missing values and how bad is it? there
Does the target column look balanced or skewed?
What looks messy at first glance?

In [12]:
df.isnull().sum()[df.isnull().sum() > 0]

max_glu_serum    96420
A1Cresult        84748
dtype: int64

In [ ]:
# These aren't missing, the test just wasn't performed
df['max_glu_serum'] = df['max_glu_serum'].fillna('None')
df['A1Cresult'] = df['A1Cresult'].fillna('None')

# Confirmation
print(df['max_glu_serum'].value_counts())
print(df['A1Cresult'].value_counts())

max_glu_serum
None    96420
Norm     2597
>200     1485
>300     1264
Name: count, dtype: int64
A1Cresult
None    84748
>8       8216
Norm     4990
>7       3812
Name: count, dtype: int64


### Missing Value Decisions for the above cell
- `max_glu_serum`: 96% missing — test not performed, filled with 'None'
- `A1Cresult`: 85% missing — test not performed, filled with 'None'

In [14]:
# Check for ? placeholders
(df == '?').sum()[( df == '?').sum() > 0]

race                  2273
weight               98569
payer_code           40256
medical_specialty    49949
diag_1                  21
diag_2                 358
diag_3                1423
dtype: int64

In [15]:
df.replace('?', np.nan, inplace=True)

In [16]:
# race — 2,273 missing, small enough to fill
df['race'] = df['race'].fillna('Unknown')

# weight — 98,569 missing (~98%) — too sparse to be useful, drop it
df.drop(columns=['weight'], inplace=True)

# payer_code — 40,256 missing, not critical to readmission prediction, drop it
df.drop(columns=['payer_code'], inplace=True)

# medical_specialty — 49,949 missing, fill with Unknown
df['medical_specialty'] = df['medical_specialty'].fillna('Unknown')

# diag_1, diag_2, diag_3 — diagnosis codes, small number missing, fill with Unknown
df['diag_1'] = df['diag_1'].fillna('Unknown')
df['diag_2'] = df['diag_2'].fillna('Unknown')
df['diag_3'] = df['diag_3'].fillna('Unknown')

In [17]:
print("Remaining nulls:", df.isnull().sum().sum())
print("Shape:", df.shape)

Remaining nulls: 0
Shape: (101766, 48)


### ? Placeholder Decisions
- `race`: 2% missing — filled with 'Unknown'
- `weight`: ~98% missing — dropped, too sparse to be useful
- `payer_code`: ~40% missing — dropped, not relevant to readmission
- `medical_specialty`: ~50% missing — filled with 'Unknown'
- `diag_1/2/3`: small number missing — filled with 'Unknown'

In [19]:
df.to_csv('C:/Users/aglaf/Documents/data engineering passion projects/projects/healthcare readmission analysis/data/processed/diabetes_cleaned.csv', index=False)
print("Cleaned data saved.")

Cleaned data saved.


## Day 3 Summary
- Replaced all `?` placeholders with NaN
- Filled missing values: race, medical_specialty, diag_1/2/3, max_glu_serum, A1Cresult
- Dropped columns: weight (98% missing), payer_code (not relevant)
- Final shape: 101,766 rows x 48 columns
- Remaining nulls: 0
- Saved cleaned dataset to data/processed/

## Day 4 — Data Cleaning Part 2
Goals:
- Load cleaned data from processed folder
- Convert data types
- Rename columns (clean naming)
- Standardize formats

In [20]:
df = pd.read_csv('C:/Users/aglaf/Documents/data engineering passion projects/projects/healthcare readmission analysis/data/processed/diabetes_cleaned.csv')
df.head()

,encounter_id,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,medical_specialty,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),6,25,1,1,Pediatrics-Endocrinology,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),1,1,7,3,Unknown,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),1,1,7,2,Unknown,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),1,1,7,2,Unknown,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),1,1,7,1,Unknown,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [21]:
df.dtypes

encounter_id                 int64
patient_nbr                  int64
race                        object
gender                      object
age                         object
admission_type_id            int64
discharge_disposition_id     int64
admission_source_id          int64
time_in_hospital             int64
medical_specialty           object
num_lab_procedures           int64
num_procedures               int64
num_medications              int64
number_outpatient            int64
number_emergency             int64
number_inpatient             int64
diag_1                      object
diag_2                      object
diag_3                      object
number_diagnoses             int64
max_glu_serum               object
A1Cresult                   object
metformin                   object
repaglinide                 object
nateglinide                 object
chlorpropamide              object
glimepiride                 object
acetohexamide               object
glipizide           

In [22]:
df.select_dtypes(include='object').columns

Index(['race', 'gender', 'age', 'medical_specialty', 'diag_1', 'diag_2',
       'diag_3', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide',
       'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide',
       'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone',
       'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide',
       'examide', 'citoglipton', 'insulin', 'glyburide-metformin',
       'glipizide-metformin', 'glimepiride-pioglitazone',
       'metformin-rosiglitazone', 'metformin-pioglitazone', 'change',
       'diabetesMed', 'readmitted'],
      dtype='object')

In [23]:
df.select_dtypes(include='object').nunique().sort_values()

citoglipton                   1
examide                       1
tolbutamide                   2
change                        2
metformin-pioglitazone        2
metformin-rosiglitazone       2
glimepiride-pioglitazone      2
glipizide-metformin           2
troglitazone                  2
diabetesMed                   2
acetohexamide                 2
tolazamide                    3
readmitted                    3
A1Cresult                     3
gender                        3
max_glu_serum                 3
glyburide-metformin           4
insulin                       4
chlorpropamide                4
metformin                     4
miglitol                      4
glimepiride                   4
rosiglitazone                 4
pioglitazone                  4
repaglinide                   4
glyburide                     4
glipizide                     4
nateglinide                   4
acarbose                      4
race                          6
age                          10
medical_

In [25]:
cat_columns = [
    'change', 'diabetesMed', 'gender', 'readmitted', 'A1Cresult',
    'max_glu_serum', 'insulin', 'metformin', 'repaglinide', 'nateglinide',
    'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide',
    'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
    'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton',
    'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone',
    'metformin-rosiglitazone', 'metformin-pioglitazone'
]

df[cat_columns] = df[cat_columns].astype('category')

# race, medical_specialty, diag_1, diag_2, diag_3
# These will be handled during feature engineering on Day 8

In [26]:
df.dtypes.value_counts()

int64       13
category    13
category     7
object       6
category     2
category     1
category     1
category     1
category     1
category     1
category     1
category     1
Name: count, dtype: int64

### Note
I made a mistake. At first i ran `df.dtypes.value_counts()` but it returned fragmented category counts instead of 
a consolidated summary. Switched to `df.dtypes.astype(str).value_counts()` to convert 
dtypes to strings first, allowing pandas to group them correctly as seen below.

In [27]:
df.dtypes.astype(str).value_counts()

category    29
int64       13
object       6
Name: count, dtype: int64

### Data Type Conversions
- Converted 29 low-cardinality columns to `category` type
- 13 numeric (int64) columns unchanged
- 6 object columns left as-is (diag codes, race, medical_specialty)
  - Will be handled during feature engineering on Day 8

In [28]:
df.columns.tolist()

['encounter_id',
 'patient_nbr',
 'race',
 'gender',
 'age',
 'admission_type_id',
 'discharge_disposition_id',
 'admission_source_id',
 'time_in_hospital',
 'medical_specialty',
 'num_lab_procedures',
 'num_procedures',
 'num_medications',
 'number_outpatient',
 'number_emergency',
 'number_inpatient',
 'diag_1',
 'diag_2',
 'diag_3',
 'number_diagnoses',
 'max_glu_serum',
 'A1Cresult',
 'metformin',
 'repaglinide',
 'nateglinide',
 'chlorpropamide',
 'glimepiride',
 'acetohexamide',
 'glipizide',
 'glyburide',
 'tolbutamide',
 'pioglitazone',
 'rosiglitazone',
 'acarbose',
 'miglitol',
 'troglitazone',
 'tolazamide',
 'examide',
 'citoglipton',
 'insulin',
 'glyburide-metformin',
 'glipizide-metformin',
 'glimepiride-pioglitazone',
 'metformin-rosiglitazone',
 'metformin-pioglitazone',
 'change',
 'diabetesMed',
 'readmitted']

In [29]:
df.rename(columns={
    'A1Cresult': 'a1c_result',
    'patient_nbr': 'patient_number',
    'glyburide-metformin': 'glyburide_metformin',
    'glipizide-metformin': 'glipizide_metformin',
    'glimepiride-pioglitazone': 'glimepiride_pioglitazone',
    'metformin-rosiglitazone': 'metformin_rosiglitazone',
    'metformin-pioglitazone': 'metformin_pioglitazone'
}, inplace=True)

# Confirm
df.columns.tolist()

['encounter_id',
 'patient_number',
 'race',
 'gender',
 'age',
 'admission_type_id',
 'discharge_disposition_id',
 'admission_source_id',
 'time_in_hospital',
 'medical_specialty',
 'num_lab_procedures',
 'num_procedures',
 'num_medications',
 'number_outpatient',
 'number_emergency',
 'number_inpatient',
 'diag_1',
 'diag_2',
 'diag_3',
 'number_diagnoses',
 'max_glu_serum',
 'a1c_result',
 'metformin',
 'repaglinide',
 'nateglinide',
 'chlorpropamide',
 'glimepiride',
 'acetohexamide',
 'glipizide',
 'glyburide',
 'tolbutamide',
 'pioglitazone',
 'rosiglitazone',
 'acarbose',
 'miglitol',
 'troglitazone',
 'tolazamide',
 'examide',
 'citoglipton',
 'insulin',
 'glyburide_metformin',
 'glipizide_metformin',
 'glimepiride_pioglitazone',
 'metformin_rosiglitazone',
 'metformin_pioglitazone',
 'change',
 'diabetesMed',
 'readmitted']

In [ ]:
df.rename(columns={
    'diabetesMed': 'diabetes_med',
    'readmitted': 'readmitted'  
}, inplace=True)

# Confirm
df.columns.tolist()

['encounter_id',
 'patient_number',
 'race',
 'gender',
 'age',
 'admission_type_id',
 'discharge_disposition_id',
 'admission_source_id',
 'time_in_hospital',
 'medical_specialty',
 'num_lab_procedures',
 'num_procedures',
 'num_medications',
 'number_outpatient',
 'number_emergency',
 'number_inpatient',
 'diag_1',
 'diag_2',
 'diag_3',
 'number_diagnoses',
 'max_glu_serum',
 'a1c_result',
 'metformin',
 'repaglinide',
 'nateglinide',
 'chlorpropamide',
 'glimepiride',
 'acetohexamide',
 'glipizide',
 'glyburide',
 'tolbutamide',
 'pioglitazone',
 'rosiglitazone',
 'acarbose',
 'miglitol',
 'troglitazone',
 'tolazamide',
 'examide',
 'citoglipton',
 'insulin',
 'glyburide_metformin',
 'glipizide_metformin',
 'glimepiride_pioglitazone',
 'metformin_rosiglitazone',
 'metformin_pioglitazone',
 'change',
 'diabetes_med',
 'readmitted']

In [31]:
df.to_csv('C:/Users/aglaf/Documents/data engineering passion projects/projects/healthcare readmission analysis/data/processed/diabetes_cleaned.csv', index=False)
print("Saved.")

Saved.


## Day 4 Summary
- Loaded cleaned data from processed folder
- Converted 29 low-cardinality columns to `category` type
- Renamed columns for consistency:
  - `A1Cresult` → `a1c_result`
  - `patient_nbr` → `patient_number`
  - `diabetesMed` → `diabetes_med`
  - Hyphenated medication columns → underscores
- Dataset is now fully clean and consistent
- Ready for EDA